# 03 · Roof features: sunlight, tree shade, shape, crowding and solar potential

**Run this in Google Colab.** (Optional: Runtime → Change runtime type → CPU + High-RAM.)

What it does:
1. **Sunlight:** downloads Global Solar Atlas PVOUT (kWh a 1 kW solar system makes per year, ~1 km pixels) and cuts out Kerala.
2. **Tree cover:** downloads ESA WorldCover 2021 (10 m) and makes a raster of **% trees in every ~30 m x 30 m block**.
3. **Sedona raster + vector join:** reads both rasters as tiles and gives every roof its sunlight and tree-cover value. *(Module 3)*
4. **Shape + crowding:** roof perimeter, compactness, and number of buildings in the same **H3 hexagon** (~0.1 km²). *(Module 3)*
5. **Solar potential per roof:** usable area → kW of panels → kWh per year. *(Module 4: environment and economics)*
6. Saves everything split by district, plus a district table.

**How to run:** Runtime → Run all. Allow Google Drive access. It takes about 20–40 minutes.

Everything printed is also saved to `MyDrive/rooftop_solar/logs/03_log.txt`. If the tab closes, send Claude that file.

In [ ]:
# Step 1: connect Google Drive, set folders, copy the clean buildings to the fast Colab disk
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, time

PROJECT = '/content/drive/MyDrive/rooftop_solar'
RAW = f'{PROJECT}/data/raw'
CLEAN = f'{PROJECT}/data/clean'
RESULTS = f'{PROJECT}/results'
LOGS = f'{PROJECT}/logs'
LOCAL = '/content/work'      # fast Colab disk, deleted when the session ends
RASTERS = f'{LOCAL}/rasters'
for folder in (RESULTS, LOGS, LOCAL, RASTERS):
    os.makedirs(folder, exist_ok=True)

LOG_FILE = f'{LOGS}/03_log.txt'
open(LOG_FILE, 'w').close()


def log(*parts):
    """Print, and also save to Google Drive so results survive if the tab closes."""
    text = ' '.join(str(p) for p in parts)
    print(text)
    with open(LOG_FILE, 'a') as f:
        f.write(text + '\n')


def log_table(pdf, csv_name=None):
    """Log a small pandas table; optionally save it as a CSV in results/."""
    log(pdf.to_string(index=False))
    if csv_name:
        pdf.to_csv(f'{RESULTS}/{csv_name}', index=False)


t = time.time()
shutil.copytree(f'{CLEAN}/buildings_clean', f'{LOCAL}/buildings_clean', dirs_exist_ok=True)
log(f'Clean buildings copied to {LOCAL} in {time.time() - t:.0f} s')

In [ ]:
# Step 2: install Java 17, PySpark 4.1, Apache Sedona 1.9.1 and rasterio (about 2-5 minutes)
import subprocess


def java_version():
    java_bin = os.path.join(os.environ['JAVA_HOME'], 'bin', 'java') if 'JAVA_HOME' in os.environ else 'java'
    out = subprocess.run(f'{java_bin} -version 2>&1', shell=True, capture_output=True, text=True).stdout
    return out.splitlines()[0] if out else 'no java'


if not any(f'version "{v}' in java_version() for v in ('17', '21')):
    !apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
!pip install -q pyspark==4.1.3 apache-sedona==1.9.1 rasterio
log('Java:', java_version())

# Sedona's Java files, downloaded directly. Spark 4.1 ships its own copy of the JTS geometry library,
# so Sedona must sit on the same class path as Spark (see Step 3) instead of using spark.jars.packages.
import urllib.request

JARS = '/content/jars'
MAVEN = 'https://repo1.maven.org/maven2'
os.makedirs(JARS, exist_ok=True)
JAR_FILES = []
for url in (f'{MAVEN}/org/apache/sedona/sedona-spark-shaded-4.1_2.13/1.9.1/sedona-spark-shaded-4.1_2.13-1.9.1.jar',
            f'{MAVEN}/org/datasyslab/geotools-wrapper/1.9.1-33.5/geotools-wrapper-1.9.1-33.5.jar'):
    path = f'{JARS}/{url.rsplit("/", 1)[1]}'
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    JAR_FILES.append(path)
log('Sedona JARs:', [os.path.basename(p) for p in JAR_FILES])

In [ ]:
# Step 3: start Spark with Sedona, using all CPU cores and most of the RAM
import psutil
from sedona.spark import SedonaContext

CORES = os.cpu_count()
MEM_GB = max(2, int(psutil.virtual_memory().total / 1e9 * 0.6))   # leave room for Python and the OS

config = (
    SedonaContext.builder()
    .master('local[*]')
    .appName('rooftop_solar')
    .config('spark.driver.memory', f'{MEM_GB}g')
    .config('spark.sql.shuffle.partitions', str(CORES * 4))
    .config('spark.driver.extraClassPath', ':'.join(JAR_FILES))
    .config('spark.sql.maxConcurrentOutputFileWriters', '20')   # write districts without sorting first
    .getOrCreate()
)
sedona = SedonaContext.create(config)
log(f'Spark {sedona.version} | CPU cores: {CORES} | Spark memory: {MEM_GB} GB')

In [ ]:
# Step 4: sunlight - Global Solar Atlas PVOUT (average kWh per kWp per day), cut to the Kerala box
import urllib.request, zipfile
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window, from_bounds

XMIN, XMAX, YMIN, YMAX = 74.85, 77.45, 8.15, 12.80     # same box as notebook 01

zip_path = f'{LOCAL}/pvout_world.zip'
if not os.path.exists(zip_path):
    urllib.request.urlretrieve('https://api.globalsolaratlas.info/download/World/'
                               'World_PVOUT_GISdata_LTAy_AvgDailyTotals_GlobalSolarAtlas-v2_GEOTIFF.zip', zip_path)
with zipfile.ZipFile(zip_path) as z:
    tif_name = next(n for n in z.namelist() if n.upper().endswith('PVOUT.TIF'))
    z.extract(tif_name, LOCAL)

with rasterio.open(f'{LOCAL}/{tif_name}') as src:
    window = from_bounds(XMIN, YMIN, XMAX, YMAX, src.transform).round_offsets().round_lengths()
    pvout = src.read(1, window=window, boundless=True, fill_value=np.nan).astype('float32')
    pvout_transform = src.window_transform(window)
    pvout_crs = src.crs
    if src.nodata is not None:
        pvout[pvout == src.nodata] = np.nan

with rasterio.open(f'{RASTERS}/pvout_kerala.tif', 'w', driver='GTiff', dtype='float32', count=1,
                   width=pvout.shape[1], height=pvout.shape[0], crs=pvout_crs,
                   transform=pvout_transform, nodata=np.nan, compress='lzw') as dst:
    dst.write(pvout, 1)

yearly = pvout[np.isfinite(pvout)] * 365.25
log(f'Sunlight raster: {pvout.shape[1]} x {pvout.shape[0]} pixels | PVOUT {yearly.min():.0f}-{yearly.max():.0f} '
    f'kWh per kWp per year (mean {yearly.mean():.0f})')

In [ ]:
# Step 5: tree cover - ESA WorldCover 2021 (10 m, class 10 = trees).
# For every 3 x 3 block of pixels (about 28 m x 28 m) store the % of pixels that are trees.
WC_URL = 'https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_{}_Map.tif'
TILES = ['N06E075', 'N09E072', 'N09E075', 'N12E072', 'N12E075']   # 3 x 3 degree tiles covering the box
PIX = 1 / 12000            # WorldCover pixel size in degrees
BLOCK = 3                  # 3 x 3 WorldCover pixels -> one output pixel
RES = PIX * BLOCK
width, height = round((XMAX - XMIN) / RES), round((YMAX - YMIN) / RES)
tree_pct = np.full((height, width), 255, dtype=np.uint8)          # 255 = no data

t = time.time()
for tile in TILES:
    path = f'{LOCAL}/{tile}.tif'
    if not os.path.exists(path):
        urllib.request.urlretrieve(WC_URL.format(tile), path)
    with rasterio.open(path) as src:
        left, bottom, right, top = src.bounds
        x0, x1, y0, y1 = max(XMIN, left), min(XMAX, right), max(YMIN, bottom), min(YMAX, top)
        if x0 >= x1 or y0 >= y1:
            continue
        c0, c1 = round((x0 - XMIN) / RES), round((x1 - XMIN) / RES)       # output columns
        r0, r1 = round((YMAX - y1) / RES), round((YMAX - y0) / RES)       # output rows
        col_off = round((XMIN + c0 * RES - left) / PIX)                    # matching tile column
        row_off = round((top - (YMAX - r0 * RES)) / PIX)                   # matching tile row
        for rs in range(r0, r1, 500):                                      # 500 output rows at a time
            re = min(rs + 500, r1)
            cls = src.read(1, window=Window(col_off, row_off + (rs - r0) * BLOCK,
                                            (c1 - c0) * BLOCK, (re - rs) * BLOCK))
            trees = (cls == 10).reshape(re - rs, BLOCK, c1 - c0, BLOCK).sum(axis=(1, 3), dtype=np.uint16)
            tree_pct[rs:re, c0:c1] = (trees * 100 + 4) // (BLOCK * BLOCK)
    log(f'  WorldCover tile {tile} done')

with rasterio.open(f'{RASTERS}/tree_pct_kerala.tif', 'w', driver='GTiff', dtype='uint8', count=1,
                   width=width, height=height, crs='EPSG:4326', transform=from_origin(XMIN, YMAX, RES, RES),
                   nodata=255, compress='lzw', tiled=True, blockxsize=512, blockysize=512) as dst:
    dst.write(tree_pct, 1)
land = tree_pct[tree_pct != 255]
log(f'Tree raster: {width} x {height} pixels of ~28 m | average tree cover in the box {land.mean():.1f}% '
    f'| {time.time() - t:.0f} s')
del tree_pct, land

In [ ]:
# Step 6: Sedona - give every roof its sunlight, tree cover, shape and crowding values
clean = sedona.read.format('geoparquet').load(f'{LOCAL}/buildings_clean').filter('keep')
clean.createOrReplaceTempView('clean')
ROOFS = clean.count()
log(f'Roofs kept after cleaning: {ROOFS:,}')

# Rasters are read as 512 x 512 tiles (one row per tile), so Spark can work on them in parallel
for name in ('tree_pct', 'pvout'):
    tiles = (sedona.read.format('raster').option('tileWidth', '512').option('tileHeight', '512')
             .load(f'{RASTERS}/{name}_kerala.tif'))
    tiles.createOrReplaceTempView(f'{name}_tiles')
    log(f'{name} raster: {tiles.count():,} tiles')

# Crowding: number of buildings in the same H3 hexagon (resolution 9, about 0.1 km2)
hex_counts = sedona.sql("""
    SELECT ST_H3CellIDs(ST_Centroid(geometry), 9, false)[0] AS h3_9, count(*) AS bldgs_in_hex
    FROM clean GROUP BY 1
""").cache()
hex_counts.createOrReplaceTempView('hex_counts')
log(f'H3 hexagons with buildings: {hex_counts.count():,}')

USABLE_SHARE = 0.6    # share of the roof that can hold panels (assumption; edges, tanks, stairs excluded)
M2_PER_KWP = 10       # roof area needed for 1 kW of panels (MNRE rule of thumb)

t0 = time.time()
features = sedona.sql(f"""
    WITH b AS (
        SELECT id, district, source, confidence, low_confidence, area_m2, geometry,
               ST_Centroid(geometry) AS centre,
               ST_LengthSpheroid(ST_Boundary(geometry)) AS perimeter_m     -- outline length in metres
        FROM clean
    ),
    with_tree AS (        -- raster + vector join: which tree-cover tile holds the roof centre?
        SELECT /*+ BROADCAST(t) */ b.*, RS_Value(t.rast, b.centre) AS tree_raw
        FROM b JOIN tree_pct_tiles t ON RS_Intersects(t.rast, b.centre)
    ),
    with_sun AS (         -- the same for the sunlight raster
        SELECT /*+ BROADCAST(p) */ w.*, RS_Value(p.rast, w.centre) * 365.25 AS pvout_kwh_per_kwp
        FROM with_tree w JOIN pvout_tiles p ON RS_Intersects(p.rast, w.centre)
    )
    SELECT /*+ BROADCAST(h) */
           s.id, s.district, s.source, s.confidence, s.low_confidence,
           s.area_m2, s.perimeter_m,
           try_divide(4 * pi() * s.area_m2, s.perimeter_m * s.perimeter_m) AS compactness,  -- 1 = circle
           CASE WHEN s.tree_raw <= 100 THEN s.tree_raw END              AS tree_pct,
           s.pvout_kwh_per_kwp,
           h.h3_9, h.bldgs_in_hex,
           s.area_m2 * {USABLE_SHARE}                                    AS usable_m2,
           s.area_m2 * {USABLE_SHARE} / {M2_PER_KWP}                     AS kwp,
           s.area_m2 * {USABLE_SHARE} / {M2_PER_KWP} * s.pvout_kwh_per_kwp AS kwh_year,
           s.geometry
    FROM with_sun s
    JOIN hex_counts h ON h.h3_9 = ST_H3CellIDs(s.centre, 9, false)[0]
""")
features.write.format('geoparquet').partitionBy('district').mode('overwrite').save(f'{LOCAL}/roof_features')
log(f'Features computed and saved in {(time.time() - t0) / 60:.1f} min')

In [ ]:
# Step 7: checks + district solar potential table (send these to Claude)
feat = sedona.read.format('geoparquet').load(f'{LOCAL}/roof_features')
feat.createOrReplaceTempView('feat')

checks = sedona.sql(f"""
    SELECT {ROOFS} AS roofs_in, count(*) AS roofs_out, count(DISTINCT id) AS distinct_ids,
           sum(CAST(tree_pct IS NULL AS INT))          AS missing_tree,
           sum(CAST(pvout_kwh_per_kwp IS NULL AS INT)) AS missing_sun
    FROM feat
""").toPandas()
log('\nCHECKS (roofs_in = roofs_out = distinct_ids, missing = 0 is ideal)')
log_table(checks)

district_potential = sedona.sql("""
    SELECT district,
           count(*)                                   AS roofs,
           round(avg(tree_pct), 1)                    AS avg_tree_pct,
           round(avg(pvout_kwh_per_kwp))              AS pvout_kwh_per_kwp,
           round(percentile_approx(bldgs_in_hex, 0.5)) AS median_bldgs_per_hex,
           round(sum(kwp) / 1e6, 2)                   AS potential_gw,
           round(sum(kwh_year) / 1e9, 2)              AS potential_twh_per_year
    FROM feat
    GROUP BY district
    ORDER BY potential_twh_per_year DESC
""").toPandas()
log('\nSOLAR POTENTIAL BY DISTRICT (all kept roofs, no shading removed yet)')
log_table(district_potential, 'district_potential.csv')

log('\nFEATURE RANGES (5th / 50th / 95th percentile)')
log_table(sedona.sql("""
    SELECT 'area_m2' AS feature, percentile_approx(area_m2, array(0.05, 0.5, 0.95)) AS p FROM feat
    UNION ALL SELECT 'compactness', percentile_approx(compactness, array(0.05, 0.5, 0.95)) FROM feat
    UNION ALL SELECT 'tree_pct', percentile_approx(tree_pct, array(0.05, 0.5, 0.95)) FROM feat
    UNION ALL SELECT 'pvout_kwh_per_kwp', percentile_approx(pvout_kwh_per_kwp, array(0.05, 0.5, 0.95)) FROM feat
    UNION ALL SELECT 'bldgs_in_hex', percentile_approx(bldgs_in_hex, array(0.05, 0.5, 0.95)) FROM feat
""").toPandas())

In [ ]:
# Step 8: copy the roof features and the two rasters to Google Drive
t = time.time()
FEATURES = f'{PROJECT}/data/features'
shutil.rmtree(f'{FEATURES}/roof_features', ignore_errors=True)    # remove an older copy first
shutil.copytree(f'{LOCAL}/roof_features', f'{FEATURES}/roof_features')
for name in ('pvout_kerala.tif', 'tree_pct_kerala.tif'):
    shutil.copy(f'{RASTERS}/{name}', f'{RAW}/{name}')
log(f'Saved roof features and rasters to Drive in {time.time() - t:.0f} s')
log('ALL DONE')

## Done

Send Claude the text of `MyDrive/rooftop_solar/logs/03_log.txt` (Drive → rooftop_solar → logs).